In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xgboost as xgb
import shap
from sklearn.inspection import permutation_importance
from statsmodels.stats.multitest import multipletests
pd.set_option('display.max_columns', None)

In [ ]:
enroll = pd.read_excel("enroll.xlsx", header=1)
enroll.head()

In [ ]:
snps = pd.read_csv("all_SNP.csv")
snps.head()

In [ ]:
profile = pd.read_excel("1. profile.xlsx")
profile.head()

In [ ]:
profile_new = profile.rename(columns={'subjid': 'FID'})
enroll_new = enroll.rename(columns={'subjid': 'FID'})

In [ ]:
df_temp = pd.merge(enroll_new, snps,on='FID')
df_snps = pd.merge(df_temp, profile_new,on='FID')
df_snps.head()

In [ ]:
df_analisis = df_snps[['FID','age','sex','isced','motscore','depscore','irascore','psyscore','aptscore','exfscore','caghigh','caglow','ccdepage', 'ccirbage', 'ccvabage', 'ccaptage', 
    'ccpobage', 'ccpsyage', 'cccogage','sxsubjm','race','region','hdiss_stage_imp','mmsetotal','sdmt1','verfct5','scnt1','swrt1',
    "19_45409113_C_T_T",
    "19_45409167_C_G_C",
    "19_45409283_C_A_A",
    "19_45409579_C_T_T",
    "19_45409988_C_T_T",
    "19_45410002_G_A_A",
    "19_45410273_G_A_A",
    "19_45410444_G_A_A",
    "19_45410548_A_G_G",
    "19_45410911_T_G_G",
    "19_45411110_T_C_C",
    "19_45411722_G_A_A",
    "19_45411941_T_C_C",
    "19_45412040_C_T_T",
    "19_45412079_C_T_T"]]


df_analisis.head()

In [ ]:
cols_ohe = [
    'region', 'sex', 'race',
    'sxsubjm'
]

df_analisis = df_analisis.replace('>70', 71)
df_analisis = df_analisis.replace('>28', 29)
cols_numericas = df_analisis.select_dtypes(include=['number']).columns
df_analisis[cols_numericas] = df_analisis[cols_numericas].where(df_analisis[cols_numericas] < 150)

df_analisis_ohe = pd.get_dummies(df_analisis, columns= cols_ohe, drop_first=False)
df_analisis_ohe.columns

In [ ]:
df_analisis_ohe["age"] = pd.to_numeric(df_analisis_ohe["age"], errors="coerce")
df_grouped = df_analisis_ohe.groupby("FID", as_index=False).mean(numeric_only=True)
df_grouped.columns

In [ ]:
condiciones = [
    df_grouped["caghigh"] < 28,
    (df_grouped["caghigh"] >= 28) & (df_grouped["caghigh"] <= 35),
    df_grouped["caghigh"] > 35
]

valores = [0, 1, 2]

df_grouped["caghigh_premut"] = np.select(condiciones, valores)

condiciones = [
    df_grouped["caghigh"] < 28,
    df_grouped["caghigh"] >= 28
]

valores = [0,1]

df_grouped["caghigh_mut"] = np.select(condiciones, valores)

condiciones = [
    df_grouped["caglow"] < 28,
    df_grouped["caglow"] >= 28
]

valores = [0,1]

df_grouped["caglow_mut"] = np.select(condiciones, valores)

df_grouped["mmdemencia"] = np.where(
    df_grouped["mmsetotal"] < 24,
    1,
    0
)

df_grouped["mmsetotal"].describe()

In [ ]:
import numpy as np
import pandas as pd
import xgboost as xgb
import shap
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.metrics import mean_squared_error


df_grouped = df_grouped.dropna(subset=["mmsetotal"])

y = df_grouped["mmsetotal"]
X = df_grouped[[
    'age',
    'sex_f', 'sex_m',
    'isced',
    'hdiss_stage_imp',
    'motscore',
    'depscore','irascore','psyscore','aptscore','exfscore',
    'caghigh','caglow',
    'ccdepage', 'ccirbage', 'ccvabage', 'ccaptage', 'ccpobage', 'ccpsyage', 'cccogage',
    'sxsubjm_1.0','sxsubjm_2.0', 'sxsubjm_3.0', 'sxsubjm_4.0', 'sxsubjm_5.0','sxsubjm_6.0',
    'race_1', 'race_2','race_3', 'race_6', 'race_8', 'race_15', 'race_16',
    'region_Australasia', 'region_Europe', 'region_Latin America','region_Northern America',
    "19_45409113_C_T_T",
    "19_45409167_C_G_C",
    "19_45409283_C_A_A",
    "19_45409579_C_T_T",
    "19_45409988_C_T_T",
    "19_45410002_G_A_A",
    "19_45410273_G_A_A",
    "19_45410444_G_A_A",
    "19_45410548_A_G_G",
    "19_45410911_T_G_G",
    "19_45411110_T_C_C",
    "19_45411722_G_A_A",
    "19_45411941_T_C_C",
    "19_45412040_C_T_T",
    "19_45412079_C_T_T"
]]


model = xgb.XGBRegressor(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)


kf = KFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = cross_val_score(
    model,
    X,
    y,
    scoring="neg_root_mean_squared_error",
    cv=kf
)

rmse_scores = -cv_scores

print("Cross-Validation RMSE por fold:", rmse_scores)
print("RMSE medio:", rmse_scores.mean())
print("RMSE std:", rmse_scores.std())


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print("Test RMSE:", rmse)


explainer = shap.Explainer(model)
shap_values = explainer(X)

# Summary plot (impacto global)
shap.summary_plot(shap_values, X)

# Feature importance media absoluta
shap_importance = np.abs(shap_values.values).mean(axis=0)
importance_df = pd.DataFrame({
    "feature": X.columns,
    "mean_abs_shap": shap_importance
}).sort_values(by="mean_abs_shap", ascending=False)

print(importance_df)

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
import numpy as np

param_dist = {
    'n_estimators': [100, 200, 500],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 5, 7, 10],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'gamma': [0, 0.1, 0.2]
}

xgb_base = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')

random_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_dist,
    n_iter=20, 
    scoring='roc_auc',
    cv=StratifiedKFold(n_splits=5),
    verbose=1,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

#Mejor modelo
best_xgb = random_search.best_estimator_
print(f"Mejores parámetros encontrados: {random_search.best_params_}")

In [ ]:
import numpy as np
import pandas as pd
import xgboost as xgb
import shap
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, confusion_matrix

df_grouped = df_grouped.dropna(subset=["mmdemencia"])

y = df_grouped["mmdemencia"]
X = df_grouped[[
    'age', 'sex_f', 'sex_m', 'isced', 'hdiss_stage_imp',
    'depscore','irascore','psyscore','aptscore','exfscore', 'motscore',
    'caghigh','caglow',
    'ccdepage', 'ccirbage', 'ccvabage', 'ccaptage', 'ccpobage', 'ccpsyage', 'cccogage',
    'sxsubjm_1.0','sxsubjm_2.0', 'sxsubjm_3.0', 'sxsubjm_4.0', 'sxsubjm_5.0','sxsubjm_6.0',
    'race_1', 'race_2','race_3', 'race_6', 'race_8', 'race_15', 'race_16',
    'region_Australasia', 'region_Europe', 'region_Latin America','region_Northern America',
    "19_45409113_C_T_T", "19_45409167_C_G_C", "19_45409283_C_A_A",
    "19_45409579_C_T_T", "19_45409988_C_T_T", "19_45410002_G_A_A",
    "19_45410273_G_A_A", "19_45410444_G_A_A", "19_45410548_A_G_G",
    "19_45410911_T_G_G", "19_45411110_T_C_C", "19_45411722_G_A_A",
    "19_45411941_T_C_C", "19_45412040_C_T_T", "19_45412079_C_T_T"
]]

model = xgb.XGBClassifier(**random_search.best_params_)

kf = KFold(n_splits=5, shuffle=True, random_state=42)

cv_accuracy = cross_val_score(model, X, y, scoring="accuracy", cv=kf)
cv_auc = cross_val_score(model, X, y, scoring="roc_auc", cv=kf)

print(f"CV Accuracy: {cv_accuracy.mean():.4f} +/- {cv_accuracy.std():.4f}")
print(f"CV AUC: {cv_auc.mean():.4f} +/- {cv_auc.std():.4f}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("\n--- Reporte de Clasificación ---")
print(classification_report(y_test, y_pred))
print(f"Test AUC-ROC: {roc_auc_score(y_test, y_proba):.4f}")

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X)

shap.summary_plot(shap_values, X)

importance_df = pd.DataFrame({
    "feature": X.columns,
    "mean_abs_shap": np.abs(shap_values).mean(axis=0)
}).sort_values(by="mean_abs_shap", ascending=False)

print("\n--- Variables por SHAP ---")
print(importance_df)

In [ ]:
import scipy.stats
scipy.stats.binom_test = scipy.stats.binomtest

if not hasattr(np, 'NaN'):
    np.NaN = np.nan
if not hasattr(np, 'float'):
    np.float = float

def binom_test_integer_safe(k, n, p, alternative):
    return scipy.stats.binomtest(int(k), int(n), p=p, alternative=alternative).pvalue


import BorutaShap

BorutaShap.binom_test = binom_test_integer_safe

from xgboost import XGBClassifier 
import pandas as pd

Feature_Selector = BorutaShap.BorutaShap(model=model, importance_measure='shap', classification=True)

Feature_Selector.fit(X=X_train, y=y_train, n_trials=100, sample=False)

Feature_Selector.plot(which_features='all')

In [ ]:
from sklearn.model_selection import KFold

list_selected_features = []

kf = KFold(n_splits=5, shuffle=True, random_state=42)

for i, (train_index, val_index) in enumerate(kf.split(X_train)):
    print(f"--- Ejecutando BorutaShap en Fold {i+1} ---")
    
    X_fold = X_train.iloc[train_index]
    y_fold = y_train.iloc[train_index]
    
    Feature_Selector = BorutaShap.BorutaShap(
        model=best_xgb, 
        importance_measure='shap', 
        classification=True
    )
    
    Feature_Selector.fit(X=X_fold, y=y_fold, n_trials=50, sample=False, verbose=False)
    
    list_selected_features.append(Feature_Selector.accepted)

from collections import Counter
all_features = [item for sublist in list_selected_features for item in sublist]
feature_consistency = Counter(all_features)

print("\nConsistencia de variables:")
for feat, count in feature_consistency.items():
    print(f"{feat}: {count}/5")